# 12v3 · Retrain the Raptor recipe with our own labels (Kaggle, GPU)

**Why?** The strongest public model family in this competition is "Raptor" (dreaddevelopment):
a CoAtNet-384 backbone + 2.5D windows + per-finding attention pooling; one model scores ~0.936–0.938.
The author shared the training code and a preprocessed image corpus but **not the labels**; the training
script expects them from outside (`--labels`).

| input | source |
|---|---|
| image corpus | our own 80-slice corpus (notebook 11, kernel outputs `rsna-knee-11-corpus80-s0/1/2`) |
| training script (fp16-patched copy for T4) | `rsna-knee-src` → `third_party/raptor/train_knee.py` |
| labels | `rsna-knee-src` → `train_labels*.csv` (version below) |
| validation | the 58 gold studies (never trained on) |

**T4 note:** the original script uses bf16; T4 GPUs have no hardware bf16, so the patched copy uses
fp16 + GradScaler. `bs=4` + gradient checkpointing fit the 16 GB memory, and `--time_budget_h` stops the
run cleanly before the session limit.

**This version (12v3): Raptor-80 on labels v3**, a teacher-student round: v3 = report labels (v2) 50% + ConvNeXt 5-fold out-of-fold predictions 50%; gold 0.9014 → 0.9223. Result: gold 0.9205, but public 0.929 / private 0.915 alone, essentially the same as labels v2 on the leaderboard.

In [ ]:
# Add the project root (the folder that contains src/) to the Python path; works locally and on Kaggle.
import os, sys
from pathlib import Path

def add_project_root():
    cands = [Path.cwd(), Path.cwd().parent]
    if Path("/kaggle/input").exists():
        # os.walk with pruning: never enters the competition DICOM folders (rglob took minutes)
        for d, dirs, files in os.walk("/kaggle/input"):
            dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
            if Path(d).name == "src" and "__init__.py" in files:
                cands.append(Path(d).parent)
    for c in cands:
        if (c / "src" / "__init__.py").exists():
            sys.path.insert(0, str(c))
            return c
    raise FileNotFoundError("src/ not found")

PROJECT_ROOT = add_project_root()
print("Project root:", PROJECT_ROOT)

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
import pandas as pd

WORK = Path("/kaggle/working")
from src.config import find_input
script = find_input("third_party/raptor/train_knee.py")
shutil.copy(script, WORK / "train_knee.py")

LABELS_FILE = "train_labels_v3.csv"
EXTRA_ARGS = ["--norm", "imagenet", "--seed", "101"]   # version-specific extra arguments
TAG = "kadirov80_lbl3_s101"
labels_csv = find_input(LABELS_FILE)
lab = pd.read_csv(labels_csv)
lab.drop(columns=["is_gold"]).to_parquet(WORK / f"labels_{TAG}.parquet", index=False)
print(script, "|", labels_csv, "|", lab.shape)

## Training

| parameter | value | note |
|---|---|---|
| `--arch` | `coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k` | Raptor's main backbone |
| `--res` | 384 | the author measured −0.029 AUC at 256 |
| `--k / --k_eval` | 12 / 24 | windows per study during training / evaluation |
| `--epochs` | 8 | ~1–1.3 h per epoch on a T4; stops early if the budget runs out |
| `--topk` | 3 | the weight average (SWA) of the best 3 epochs is saved as well |

In [ ]:
cmd = [sys.executable, "train_knee.py",
       "--arch", "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k", "--res", "384",
       "--epochs", "8", "--bs", "4", "--k", "12", "--k_eval", "24", "--grad_ckpt",
       "--workers", "4", "--topk", "3", "--tag", TAG,
       "--labels", str(WORK / f"labels_{TAG}.parquet"),
       "--time_budget_h", "10.5", "--log_every", "200"] + EXTRA_ARGS
print(" ".join(cmd))
proc = subprocess.Popen(cmd, cwd=WORK, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end="")
proc.wait()
assert proc.returncode == 0, f"training failed with exit code {proc.returncode}"

In [ ]:
import json
res = json.load(open(WORK / f"raptor_ft_{TAG}.json"))
print("Best gold macro AUC:", round(res["best_gold_auc"], 4), "| epochs:", res["epochs_done"])
pd.Series(res["aucs"]).round(3).to_frame("gold AUC").T